In [98]:
import requests
import pandas as pd
import json

In [ ]:
# load data
df = pd.read_csv('reviews/scopus/scopus_flattened_annotated.csv')
df.head()

In [104]:
import re

def sanitize_title(title):
    return re.sub(r'[:&,]', '', title)     # Remove special characters that break OpenAlex filter

def fetch_openalex_by_title(title):
    base_url = "https://api.openalex.org/works"
    clean_title = sanitize_title(title)
    params = {
        "filter": f"title.search:{clean_title}",
        "per-page": 1
    }

    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        if 'results' in data and len(data['results']) > 0:
            return data['results'][0]
        return None
    except requests.RequestException as e:
        print(f"Error fetching '{title}': {e}")
        return None

# Apply to dataframe
df['openalex_res'] = df['dc:title'].apply(fetch_openalex_by_title)

In [107]:
# PARSE OpenAlex RESULTS to get relevant information

# Initialize new columns
df['is_open_access'] = None
df['is_published'] = None
df['any_repository_has_fulltext'] = None
df['corresponding_author_country'] = None
df['topics'] = None
df['fields'] = None
df['raw_type'] = None
df['indexed_in'] = None
df['type'] = None
df['has_pdf'] = None
df['relevance_score'] = None
df['citation_percentile_value'] = None
df['is_top_1_percent'] = None
df['is_top_10_percent'] = None

# Loop through rows and extract safely
for idx, row in df.iterrows():
    res = row['openalex_res']
    if res is not None:
        # Open access
        df.at[idx, 'is_open_access'] = res.get('open_access', {}).get('is_oa', False)
        
        # Published status
        df.at[idx, 'is_published'] = res.get('primary_location', {}).get('is_published', False)
        
        # Any repository fulltext
        df.at[idx, 'any_repository_has_fulltext'] = res.get('open_access', {}).get('any_repository_has_fulltext', False)
        
        # Country of corresponding author
        corresponding_country = None
        for auth in res.get('authorships', []):
            if auth.get('is_corresponding', False):
                institutions = auth.get('institutions', [])
                if institutions:
                    corresponding_country = institutions[0].get('country_code')
                break
        df.at[idx, 'corresponding_author_country'] = corresponding_country
        
        # List of topic names
        df.at[idx, 'topics'] = [t.get('display_name') for t in res.get('topics', [])]
        
        # List of field names (from topics)
        df.at[idx, 'fields'] = list({t.get('field', {}).get('display_name') 
                                    for t in res.get('topics', []) if t.get('field')})
        
        # Raw type of publication
        df.at[idx, 'raw_type'] = res.get('primary_location', {}).get('raw_type')
        
        # Indexed in (list)
        df.at[idx, 'indexed_in'] = res.get('indexed_in', [])
        
        # Overall type of the work
        df.at[idx, 'type'] = res.get('type')
        
        # Whether PDF is available
        df.at[idx, 'has_pdf'] = res.get('has_content', {}).get('pdf', False)

        # Citation normalized percentile
        cnp = res.get('citation_normalized_percentile') or {}  # <-- if None, use empty dict
        df.at[idx, 'citation_percentile_value'] = cnp.get('value', None)
        df.at[idx, 'is_top_1_percent'] = cnp.get('is_in_top_1_percent', False)
        df.at[idx, 'is_top_10_percent'] = cnp.get('is_in_top_10_percent', False)
